In [2]:
import os, shutil, random
from pathlib import Path

### Detection

In [6]:
src = Path("/home/somethink/parking_system/yolo_plate_dataset")
dst = Path("/home/somethink/parking_system/yolo_plate_data")
val_ratio = 0.2
random_seed = 42

In [ ]:
random.seed(random_seed)

images = sorted([
    f for f in src.glob("*.jpg")
    if (src / f.with_suffix(".txt").name).exists()
])
print(f"Tong so anh co label: {len(images)}")

random.shuffle(images)
n_val = int(len(images) * val_ratio)
splits = {"val": images[:n_val], "train": images[n_val:]}

for split, imgs in splits.items():
    img_dir = dst / "images" / split
    lbl_dir = dst / "labels" / split
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    for img in imgs:
        shutil.copy(img, img_dir / img.name)
        shutil.copy(src / img.with_suffix(".txt").name,
                    lbl_dir / img.with_suffix(".txt").name)
    print(f"  {split}: {len(imgs)} anh")

yaml_content = (
    f"path: {dst.resolve()}\n"
    "train: images/train\n"
    "val: images/val\n"
    "nc: 1\n"
    "names: ['plate']\n"
)
Path("plates.yaml").write_text(yaml_content)
print("\nTao file: plates.yaml")
print("Chay train:")
print("  yolo train model=yolov8n.pt data=plates.yaml epochs=100 imgsz=640 batch=16 device=0")

Tong so anh co label: 8259
  val: 1651 anh
  train: 6608 anh

Tao file: plates.yaml
Chay train:
  yolo train model=yolov8n.pt data=plates.yaml epochs=100 imgsz=640 batch=16 device=0


In [6]:
!yolo train model=yolov8n.pt data=plates.yaml epochs=35 imgsz=640 batch=4 amp=True device=0 workers=2

New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.19 🚀 Python-3.10.12 torch-2.8.0 CUDA:0 (Orin, 7620MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=plates.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=35, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train5, nbs=64, nms=False, opset=None, optimize

In [7]:
!MPLBACKEND=Agg yolo detect val model=/home/somethink/parking_system/runs/detect/train5/weights/best.pt data=plates.yaml device=0 plots=False

Ultralytics 8.4.19 🚀 Python-3.10.12 torch-2.8.0 CUDA:0 (Orin, 7620MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 482.4±140.0 MB/s, size: 42.2 KB)
val: Scanning /home/somethink/parking_system/yolo_plate_data/labels/val.cache... 1651 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1651/1651 64.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 104/104 3.7it/s 28.4s0.3s
                   all       1651       1686      0.994      0.991      0.995      0.746
Speed: 0.9ms preprocess, 10.6ms inference, 0.0ms loss, 2.0ms postprocess per image
💡 Learn more at https://docs.ultralytics.com/modes/val


### Recognition

In [3]:
# ── Config ──
ocr_src = Path("/home/somethink/parking_system/yolo_plate_ocr_dataset")  # ảnh + label gốc
ocr_dst = Path("/home/somethink/parking_system/yolo_plate_ocr_data")     # split train/val
ocr_val_ratio = 0.2

# 36 classes: 0-9 → '0'-'9', 10-35 → 'A'-'Z'
NUM_CLASSES = 36
CLASS_NAMES = [str(i) for i in range(10)] + [chr(c) for c in range(ord('A'), ord('Z') + 1)]
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")

Classes (36): ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']


In [4]:
# ── Verify dataset ──
ocr_images = sorted([
    f for f in ocr_src.glob("*.jpg")
    if (ocr_src / f.with_suffix(".txt").name).exists()
])
# Cũng check .png
ocr_images += sorted([
    f for f in ocr_src.glob("*.png")
    if (ocr_src / f.with_suffix(".txt").name).exists()
])
print(f"Tong so anh co label: {len(ocr_images)}")

# Thong ke so class thuc te trong dataset
from collections import Counter
class_counts = Counter()
bad_labels = []
for img in ocr_images:
    lbl = ocr_src / img.with_suffix(".txt").name
    for line in lbl.read_text().strip().splitlines():
        parts = line.strip().split()
        if len(parts) >= 5:
            cls_id = int(parts[0])
            class_counts[cls_id] += 1
            if cls_id < 0 or cls_id >= NUM_CLASSES:
                bad_labels.append((img.name, cls_id))

print(f"\nTong so ky tu annotated: {sum(class_counts.values())}")
print(f"So class co data: {len(class_counts)}/{NUM_CLASSES}")

if bad_labels:
    print(f"\n⚠️  {len(bad_labels)} label ngoai range [0, {NUM_CLASSES}):")
    for name, cid in bad_labels[:10]:
        print(f"  {name}: class={cid}")
else:
    print("✅ Tat ca label trong range")

# Top classes
print("\nTop 10 classes:")
for cls_id, cnt in class_counts.most_common(10):
    name = CLASS_NAMES[cls_id] if cls_id < NUM_CLASSES else f"?{cls_id}"
    print(f"  [{cls_id:2d}] {name}: {cnt}")

# Classes thiếu data
missing = [CLASS_NAMES[i] for i in range(NUM_CLASSES) if class_counts[i] == 0]
if missing:
    print(f"\n⚠️  Classes không có data: {missing}")


Tong so anh co label: 3188

Tong so ky tu annotated: 25781
So class co data: 36/36
✅ Tat ca label trong range

Top 10 classes:
  [ 1] 1: 3475
  [ 5] 5: 3270
  [ 9] 9: 2506
  [ 2] 2: 2011
  [ 4] 4: 1830
  [ 6] 6: 1802
  [ 0] 0: 1800
  [ 7] 7: 1754
  [ 3] 3: 1673
  [ 8] 8: 1569


In [7]:
# ── Split train/val ──
random.seed(random_seed)
random.shuffle(ocr_images)

n_val = int(len(ocr_images) * ocr_val_ratio)
ocr_splits = {"val": ocr_images[:n_val], "train": ocr_images[n_val:]}

for split, imgs in ocr_splits.items():
    img_dir = ocr_dst / "images" / split
    lbl_dir = ocr_dst / "labels" / split
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    for img in imgs:
        shutil.copy(img, img_dir / img.name)
        shutil.copy(ocr_src / img.with_suffix(".txt").name,
                    lbl_dir / img.with_suffix(".txt").name)
    print(f"  {split}: {len(imgs)} anh")

# ── Tao YAML ──
names_str = ", ".join([f"'{n}'" for n in CLASS_NAMES])
ocr_yaml = (
    f"path: {ocr_dst.resolve()}\n"
    f"train: images/train\n"
    f"val: images/val\n"
    f"nc: {NUM_CLASSES}\n"
    f"names: [{names_str}]\n"
)
Path("plate_ocr.yaml").write_text(ocr_yaml)
print(f"\n✅ Tao file: plate_ocr.yaml")
print(f"\nNoi dung:")
print(ocr_yaml)

  val: 637 anh
  train: 2551 anh

✅ Tao file: plate_ocr.yaml

Noi dung:
path: /home/somethink/parking_system/yolo_plate_ocr_data
train: images/train
val: images/val
nc: 36
names: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']



In [10]:
!yolo train model=yolov8n.pt data=plate_ocr.yaml epochs=100 patience=7 imgsz=320 batch=8 amp=True device=0 workers=2 project=runs/detect name=ocr_train  

New https://pypi.org/project/ultralytics/8.4.33 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.19 🚀 Python-3.10.12 torch-2.8.0 CUDA:0 (Orin, 7620MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=plate_ocr.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ocr_train2, nbs=64, nms=False, opset=None, 

In [12]:
!MPLBACKEND=Agg yolo detect val model=runs/detect/ocr_train/weights/best.pt data=plate_ocr.yaml imgsz=320 device=0 plots=False

Ultralytics 8.4.19 🚀 Python-3.10.12 torch-2.8.0 CUDA:0 (Orin, 7620MiB)
Model summary (fused): 73 layers, 3,012,668 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 517.1±313.2 MB/s, size: 90.8 KB)
val: Scanning /home/somethink/parking_system/yolo_plate_ocr_data/labels/val.cache... 637 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 637/637 31.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 40/40 6.4it/s 6.3s0.1s
                   all        637       5180      0.931      0.866      0.872      0.653
                     0        280        375      0.992      0.992      0.993      0.771
                     1        461        696      0.986      0.991      0.993      0.634
                     2        295        385      0.997      0.992      0.995      0.763
                     3        246        304      0.997      0.995      0.995      0.769
                     4    

In [9]:
import os
print(os.getpid())

49094


In [13]:
from ultralytics import YOLO
import cv2
import numpy as np

ocr_model = YOLO("runs/detect/ocr_train/weights/best.pt", task="detect")

CLASS_NAMES = [str(i) for i in range(10)] + [chr(c) for c in range(ord('A'), ord('Z') + 1)]

def read_plate(model, crop_img, conf=0.3):
    """Detect ký tự + sort theo vị trí → chuỗi biển số."""
    results = model(crop_img, imgsz=320, conf=conf, verbose=False)
    chars = []
    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            cls_id = int(box.cls)
            c = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else '?'
            chars.append({
                'char': c,
                'conf': float(box.conf),
                'cx': (x1 + x2) / 2,
                'cy': (y1 + y2) / 2,
                'bbox': (int(x1), int(y1), int(x2), int(y2)),
            })

    if not chars:
        return '', [], 0.0

    # Sort: tách 2 dòng nếu cần
    ys = [c['cy'] for c in chars]
    y_range = max(ys) - min(ys)
    img_h = crop_img.shape[0]

    if y_range > img_h * 0.3:  # 2 dòng
        y_mid = (min(ys) + max(ys)) / 2
        line1 = sorted([c for c in chars if c['cy'] < y_mid], key=lambda c: c['cx'])
        line2 = sorted([c for c in chars if c['cy'] >= y_mid], key=lambda c: c['cx'])
        sorted_chars = line1 + line2
    else:  # 1 dòng
        sorted_chars = sorted(chars, key=lambda c: c['cx'])

    text = ''.join(c['char'] for c in sorted_chars)
    avg_conf = sum(c['conf'] for c in sorted_chars) / len(sorted_chars)
    return text, sorted_chars, avg_conf

# Test vài ảnh từ val set
val_dir = ocr_dst / "images" / "val"
test_imgs = sorted(val_dir.glob('*.jpg'))[:5] + sorted(val_dir.glob('*.png'))[:5]

for img_path in test_imgs[:8]:
    img = cv2.imread(str(img_path))
    if img is None: continue
    text, chars, conf = read_plate(ocr_model, img)
    print(f'{img_path.name:30s} → {text:15s} (conf={conf:.2f}, {len(chars)} chars)')


0.jpg                          → 17571111111111  (conf=0.54, 14 chars)
10xemay1332.jpg                → 59A303615       (conf=0.90, 9 chars)
10xemay1510.jpg                → 59X169592       (conf=0.89, 9 chars)
10xemay1825.jpg                → 59U135226       (conf=0.88, 9 chars)
10xemay2052.jpg                → 85F114927       (conf=0.87, 9 chars)
